In [1]:
import numpy as np
import gurobipy
import pandas as pd

In [4]:
# --- 1. Lecture du fichier ---
df = pd.read_excel("efficacity_data_v1.xlsx")

# ============================
# PARTIE A — CALIBRATED
# ============================

df_calibrated = df[
    df["id_simulation"].astype(str).str.contains("calibrated", case=False)
][["building_name", "id_simulation", "conso_total_mwh_an"]]

# Liste des calibrated par bâtiment
calibrated_list = (
    df_calibrated
    .groupby("building_name")["conso_total_mwh_an"]
    .apply(list)
    .to_dict()
)

# ============================
# PARTIE B — SANS CALIBRATED
# ============================

df_nc = df[
    ~df["id_simulation"].astype(str).str.contains("calibrated", case=False)
]

# Colonnes utiles
df_nc = df_nc[[
    "building_name",
    "id_simulation",
    "conso_total_mwh_an",
    "cout_investissement_euros"
]]

# --- 2. Groupement ---
group_conso = df_nc.groupby("building_name")["conso_total_mwh_an"].apply(list)
group_cost  = df_nc.groupby("building_name")["cout_investissement_euros"].apply(list)

# --- 3. Taille maximale ---
max_len = max(
    group_conso.apply(len).max(),
    group_cost.apply(len).max()
)

# --- 4. Matrices à taille fixe ---
conso_matrix = np.array([
    vals + [np.inf] * (max_len - len(vals))
    for vals in group_conso
])

cost_matrix = np.array([
    vals + [np.inf] * (max_len - len(vals))
    for vals in group_cost
])

# --- 5. Métadonnées utiles ---
building_names = group_conso.index.tolist()

print("Conso matrix shape :", conso_matrix.shape)
print("Cost matrix shape  :", cost_matrix.shape)
print("Nb bâtiments       :", len(building_names))
print("Nb calibrated      :", len(df_calibrated))

Conso matrix shape : (72, 15)
Cost matrix shape  : (72, 15)
Nb bâtiments       : 72
Nb calibrated      : 76
